In [ ]:
# Localiza la raíz del repositorio subiendo desde donde se ejecute el notebook,
# para no depender de una ruta fija de una máquina concreta.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
while not ((PROJECT_DIR / "data").exists() and (PROJECT_DIR / "notebooks").exists()):
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Importación

Carga de las librerías necesarias y del dataset de features resultante de `02_feature_engineering.ipynb` (`data/processed/sevilla/listings_full_features.csv`).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(f"{PROJECT_DIR}/data/processed/sevilla/listings_full_features.csv")
df.shape

(7640, 79)

## 2. Preparación de X e y

Separar identificadores, objetivo (`price`) y features. Convertir las columnas booleanas a 0/1 y decidir qué hacer con los nulos que quedan, ya que una regresión lineal no admite `NaN` directamente.

### 2.1 Identificadores, objetivo y features

Además de los identificadores, hay que excluir de `feature_cols` las columnas calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): si se dejan dentro, el modelo no aprende ningún patrón real, deshace la fórmula y "adivina" el precio exacto. Es fuga de información, igual que en las otras cuatro ciudades.

**Fuga corregida más abajo**: `neighbourhood_price_encoded` (creada en `02_feature_engineering.ipynb`) se calculó allí usando todo el dataset, no solo lo que aquí es train. Se arregla en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (pendiente de la corrección de la
# sección 3.1bis): se excluye de X igual que los identificadores, pero se mantiene en
# df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((7640, 71), (7640,))

### 2.2 Booleanas a 0/1

40 de las 71 features son booleanas (los one-hot y los flags), menos que en Valencia (50), por tener menos distritos (11 dummies de `district_*` frente a 19) y menos categorías de `property_type` tras agrupar (13 frente a 15). `scikit-learn` las admite tal cual, pero se convierten a `int` de forma explícita.

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

40

### 2.3 Nulos restantes

`review_scores_rating`, `listing_age_days` y `days_since_last_review` son `NaN` en las 511 filas sin reviews todavía (`has_reviews == False`). Para este baseline se imputan con la mediana. Un modelo de árboles en `04_model_training.ipynb` podrá trabajar con el `NaN` directamente.

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 71 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6).

## 3. Train/test split

Reservar un conjunto de test antes de tocar nada más, y guardarlo en `data/processed/sevilla/` para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` partan del mismo split.

### 3.1 Dividir

80/20, con `random_state` fijo. Se estratifica por `room_type`: en la EDA se vio que `Shared room` (0.31%) y `Hotel room` (0.18%) tienen muestra muy escasa en Sevilla. Un split aleatorio sin más podría dejar a alguna de las dos con muy pocas filas en test por puro azar.

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((6112, 71), (1528, 71))

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

Mismo cálculo que en `02_feature_engineering.ipynb` (media de `price` por `neighbourhood_cleansed`, suavizada con la media global y `smoothing=10`), pero ahora solo con `df_train`. El mapa aprendido en train se aplica tal cual a `df_test` (un barrio de test que no apareciera en train recibe la media global de train como respaldo).

Con esto corregido, `neighbourhood_cleansed` ya cumplió su función y se descarta de `df_train`/`df_test`.

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

print("barrios de test no vistos en train:", df_test["neighbourhood_cleansed"].map(smoothed_mean_train).isnull().sum())

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

barrios de test no vistos en train: 3


,neighbourhood_price_encoded
count,6112.000000
mean,149.798241
std,23.984957
min,94.288275
25%,125.972685
50%,154.943533
75%,168.217279
max,196.438070


A diferencia de Valencia (donde los 84 barrios aparecían todos en train), aquí **3 de los 102 barrios de Sevilla** se quedan solo en test (99 en train): con más del doble de barrios que Valencia repartidos sobre un dataset más pequeño (6112 filas de train frente a ~5800), varios barrios de muestra mínima (recordar `Puerto de la Torre`/`Campanillas`-tipo, con muy pocos anuncios cada uno) tienen más probabilidad de quedar fuera de train por puro azar del split. Para esos 3 barrios, `neighbourhood_price_encoded` en test cae de vuelta a la media global de train: exactamente el respaldo para el que se diseñó el `.fillna()`.

### 3.2 Guardar el split

Se guarda `df_train`/`df_test` ya con `neighbourhood_price_encoded` corregido, pero antes de la imputación y la conversión de booleanas de la sección 2, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` puedan decidir su propio tratamiento.

In [8]:
df_train.to_csv(f"{PROJECT_DIR}/data/processed/sevilla/listings_train.csv", index=False)
df_test.to_csv(f"{PROJECT_DIR}/data/processed/sevilla/listings_test.csv", index=False)

## 4. Baseline ingenuo

Un modelo trivial (predecir siempre la media, la mediana, o la mediana por una variable de tamaño) como suelo mínimo: cualquier modelo real tiene que superar esto para que merezca la pena.

### 4.1 Predecir siempre la media

Por definición, un modelo que siempre predice la media del train tiene R² ≈ 0 sobre el test. Sirve como punto cero.

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 299.31545488415645
MAE: 88.0980228920534
R2: -0.000903964368081267


### 4.2 Predecir siempre la mediana

Dado el sesgo de `price` (skew 18.2 en la EDA), la mediana debería ser un mejor "valor típico" que la media.

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 303.34063044558184
MAE: 78.15759816753926
R2: -0.02800516299554734


El MAE mejora (78.16€ frente a 88.10€ con la media), pero el R² empeora ligeramente (-0.03 frente a ~0.00): el R² compara contra la media por definición, así que cualquier predicción distinta puede bajarlo aunque sea mejor en otros términos. Igual que en las otras ciudades, un recordatorio de que la métrica de referencia importa.

### 4.3 Predecir la mediana según `accommodates`

En la EDA/feature engineering de Sevilla la variable más relacionada con `price` es **`accommodates`** (Spearman 0.50, Pearson 0.22). Un baseline algo menos ingenuo: mediana por cada valor de `accommodates`, calculada solo con train.

In [11]:
train_medians_by_accommodates = X_train.assign(price=y_train).groupby("accommodates")["price"].median()

pred_accommodates = X_test["accommodates"].map(train_medians_by_accommodates).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_accommodates)))
print("MAE:", mean_absolute_error(y_test, pred_accommodates))
print("R2:", r2_score(y_test, pred_accommodates))

RMSE: 296.2462634143702
MAE: 70.22517670157069
R2: 0.019517406029115447


Mejora en MAE (78.16€ a 70.23€), pero el R² apenas se mueve (-0.03 a **0.02**) y el RMSE casi no cambia (303.3 a 296.2): la mejora más débil de las cinco ciudades con diferencia (en Valencia ya llegaba a 0.21). Antes de sacar conclusiones sobre el propio modelo, hace falta mirar qué hay en `y_test`: se retoma en la sección 6.2, porque la explicación no es que `accommodates` no sirva, es otra cosa.

## 5. Métricas de evaluación

Definir aquí las métricas que se van a usar de forma consistente en todo el modelado (RMSE, MAE, R², MAPE).

### 5.1 Función `evaluate`

Se añade una cuarta métrica, MAPE, el error medio en porcentaje sobre el precio real. Todo se empaqueta en una función para no repetir el código en cada modelo.

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_accommodates, "Mediana por accommodates"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,299.32,88.10,-0.00,74.72
Mediana,303.34,78.16,-0.03,48.07
Mediana por accommodates,296.25,70.23,0.02,38.28


El MAPE mejora con claridad en cada paso (74.72% → 48.07% → 38.28%), coherente y en la misma dirección que en las otras ciudades: es MAE y MAPE, no RMSE/R², los que reflejan con fidelidad la mejora real de cada baseline aquí. Se retoma por qué en la sección 6.2.

## 6. Baseline real: regresión lineal

Un primer modelo simple e interpretable, entrenado sobre `price_log` por el sesgo ya visto en la EDA.

### 6.1 Entrenar

Se entrena sobre `log1p(price)`. Las predicciones se deshacen con `expm1` antes de evaluar.

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,299.32,88.10,-0.00,74.72
Mediana,303.34,78.16,-0.03,48.07
Mediana por accommodates,296.25,70.23,0.02,38.28
Regresión lineal (log),289.82,62.55,0.06,30.25


**R²=0.06, MAE≈62.55€, MAPE≈30.3%**: el R² es, con diferencia, el más bajo de las cinco ciudades, incluso por debajo de Valencia (0.32). Pero MAE y MAPE sí mejoran de forma clara y consistente en cada paso del baseline (88.10€→78.16€→70.23€→62.55€ de MAE, 74.72%→48.07%→38.28%→30.25% de MAPE): el modelo **sí** está aprendiendo señal real, coherente con las correlaciones de Spearman ya vistas en la EDA/FE (`accommodates` 0.50, `bedrooms` 0.48). La contradicción aparente (MAE mejora con claridad, R²/RMSE casi no se mueven) tiene una causa concreta, no es un fallo del modelo: se investiga a continuación.

In [16]:
y_test.describe()

count    1528.000000
mean      157.017048
std       299.278209
min         4.780000
25%        82.022500
50%       109.505000
75%       150.500000
max      8052.000000
Name: price, dtype: float64

Ahí está la explicación: el `y_test` de Sevilla tiene una `std` de ~299€, muy por encima de su propia mediana (~110€), porque **un puñado de precios extremos de la cola alta de la EDA (sección 8.2) cayeron en el conjunto de test por el azar del split**, entre ellos, el propio "Rent it if you can afford it!" (8052€) que ya se señaló en la EDA como probable precio disuasorio sin criterio objetivo para descartarlo. RMSE y R² se calculan sobre errores al cuadrado, así que unas pocas filas con un residuo de miles de euros dominan la métrica entera, mientras que MAE y MAPE (que promedian el error absoluto, no el cuadrado) apenas se inmutan y sí reflejan la mejora real del modelo. Esto no es un error de este notebook: es la consecuencia directa, ya anticipada, de la decisión tomada en la EDA (sección 8.2) de no descartar esa cola alta al no poder aislarla con un criterio objetivo.

In [17]:
y_test.sort_values(ascending=False).head(6)

6592    8052.00
1611    3702.61
140     3510.00
1554    3490.00
1341    3000.00
5871    2187.67
Name: price, dtype: float64

6 filas por encima de 1140€ (de un test de 1528 filas, el 0.4%) explican la mayor parte de la varianza de `y_test`: quitarlas hipotéticamente del cálculo de `std` la reduciría de ~299€ a ~133€, más del doble. Ninguna de las cinco ciudades había mostrado antes un test set tan sensible a un puñado de filas: consecuencia directa de no haber una frontera objetiva de artefacto que aplicar en la EDA de Sevilla.

### 6.3 Un aviso a tener en cuenta

Al entrenar aparecen avisos de `numpy` (`divide by zero`, `overflow`... `encountered in matmul`), igual que en las otras ciudades.

In [18]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(1.6722289392405567e+19)

Un número de condición altísimo (10^19, el más alto de las cinco ciudades) indica una matriz muy mal condicionada: `X` incluye a propósito tanto la versión bruta como la versión `_log` de varias variables (`bedrooms`/`bedrooms_log`...) y varias codificaciones categóricas que se solapan. Esto no invalida las métricas (`scikit-learn` resuelve con SVD, sin `NaN`/`inf` en las predicciones), pero sí impide interpretar los coeficientes uno a uno. Se deja igual que en las otras ciudades para `04_model_training.ipynb` (modelo regularizado o selección de variables, si hiciera falta interpretar coeficientes).

## 7. Conclusiones

Resumen de los resultados del baseline.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 299.32 | 88.10 | -0.00 | 74.72 |
| Mediana | 303.34 | 78.16 | -0.03 | 48.07 |
| Mediana por `accommodates` | 296.25 | 70.23 | 0.02 | 38.28 |
| Regresión lineal (log) | 289.82 | 62.55 | 0.06 | 30.25 |

MAE y MAPE mejoran con claridad y de forma monótona en cada paso. RMSE y R² apenas se mueven, y con valores muy por debajo de las otras cuatro ciudades. La sección 6.2 encuentra la causa: un puñado de precios extremos de la cola alta de `price` (que la EDA, sección 8.2, decidió no descartar por no tener un criterio objetivo de artefacto) cayeron en el test set y dominan las métricas basadas en error al cuadrado.

### Tres problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Se excluyeron en la sección 2, igual que en las otras cuatro ciudades.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` se calculaba con todo el dataset. Se corrige en la sección 3.1bis, recalculándola solo con `df_train`, con la particularidad, única de Sevilla, de que 3 de los 102 barrios no aparecen en train y caen de vuelta a la media global.
- **Muestra escasa en `Shared room`/`Hotel room`** (24 y 14 anuncios sobre 7640): el split estratifica por `room_type`.
- **Multicolinealidad severa** en la regresión lineal, por la versión bruta y `_log` de varias variables a la vez, más marcada aquí que en las otras ciudades (número de condición 10^19). No afecta a las métricas de predicción, pero impide interpretar los coeficientes uno a uno.

### Una diferencia real con las otras ciudades, no un error de pipeline

El R² del baseline lineal en Sevilla (0.06) es, con diferencia, el más bajo de las cinco ciudades. A diferencia de Valencia (donde la brecha Pearson/Spearman ya explicaba un R² moderado de 0.32), aquí la causa principal no es la falta de linealidad de la relación tamaño-precio (las correlaciones de Spearman, 0.39-0.50, son razonables) sino una **decisión explícita tomada en la EDA**: al no encontrar un criterio objetivo para separar precios disuasorios de lujo genuino en la cola alta (sección 8.2), esa cola se mantuvo intacta, y el azar del split de train/test concentró varias de esas filas extremas en test. RMSE y R² son extremadamente sensibles a esto por construcción (error al cuadrado). MAE y MAPE, mucho más robustos, muestran una historia coherente de mejora progresiva en cada paso del baseline.

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, XGBoost, LightGBM...) tiene que superar **MAE≈62.55€ / MAPE≈30.3%** como referencia principal, más que el R²=0.06 (una métrica que aquí depende demasiado de qué filas concretas caigan en test). Un modelo de árboles no necesita la imputación por mediana de la sección 2.3, no le afecta la multicolinealidad de la sección 6.3, y su error absoluto (MAE) debería seguir bajando de forma consistente con las variables de tamaño (`accommodates`, `bedrooms`) y `neighbourhood_price_encoded`, aunque el R² siga estando limitado por la sensibilidad de Sevilla a los pocos precios extremos del test.

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/sevilla/`, con el mismo split (80/20, `random_state=42`, estratificado por `room_type`) y `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.